# 10 · Прогоны судьи по всему корпусу

Требует уже поднятого vLLM: сначала `00_setup.ipynb` целиком, включая смоук на logprobs.

Каждый прогон — отдельная ячейка с ожидаемым временем; после каждого — коммит
артефакта, чтобы обрыв сессии не стоил часа GPU. Все прогоны резюмируемы: кэш судьи
лежит на File Storage, `--resume` дочитывает уже посчитанные `id`.

| Прогон | CLI | Время на A100 |
|---|---|---|
| zero-shot, холистический | `scripts/score.py` | ~15 мин |
| few-shot | `scripts/score.py` | ~20 мин |
| разделённые оси | `scripts/run_m3.py --prompt-style axes` | ~20 мин |
| self-consistency k=8 | `scripts/run_m3.py --sc-n 8` | ~50 мин |
| пофрагментная верификация | `scripts/run_m3.py` (ждёт D1) | ~40 мин |

Ничего длиннее двух часов здесь нет — это сознательно: VM ноутбука останавливается
при простое, длинные прогоны идут через `jobs/*.yaml`.

## 0. Конфигурация и проверка, что vLLM жив

In [ ]:
# ======================= КОНФИГУРАЦИЯ — правится только здесь =======================
BASE     = "/home/jupyter/filestore/neurodrive"   # File Storage: переживает рестарт VM
REPO     = f"{BASE}/rag-reliability"
REPO_URL = "https://github.com/MurkaSelebry/rag-reliability.git"
BRANCH   = "integration"                          # НЕ qwen7b-notebook: та ветка устарела
CACHE    = f"{BASE}/cache/m3_judge"               # кэш судьи -> прогон резюмируется
LOGS     = f"{BASE}/logs"
DATA     = "data/alfa.jsonl"                      # канонический корпус, 2233 кейса
FOLDS    = "data/splits/folds_alfa.json"          # сплит только отсюда, split_samples не вызываем
MODEL    = "Qwen/Qwen2.5-7B-Instruct"
API_BASE = "http://localhost:8000/v1"
# ===================================================================================

import os, subprocess

branch = subprocess.check_output(
    ["git", "-C", REPO, "rev-parse", "--abbrev-ref", "HEAD"]
).decode().strip()
assert branch == BRANCH, (
    f"репозиторий на ветке {branch}, ожидалась {BRANCH}. Прогон с другой ветки несравним "
    "с остальными: перезапусти notebooks/00_setup.ipynb"
)
print("branch:", branch)

import requests

os.environ.setdefault("OPENAI_API_KEY", "dummy")   # vLLM ключ не проверяет, клиент требует непустой
models = requests.get(f"{API_BASE}/models", timeout=5)
assert models.ok, (
    "vLLM не отвечает на " + API_BASE + " — запусти notebooks/00_setup.ipynb "
    "(ячейка «vLLM в фоне») в этом же проекте и дождись «vLLM up»"
)
print("vLLM:", [m["id"] for m in models.json()["data"]])


## zero-shot, холистический — ~15 мин

In [ ]:
# ~15 мин на 2233 кейсах
!cd {REPO} && python scripts/score.py --method m3_openai_judge --variant zero_shot \
    --data {DATA} --model {MODEL} --resume --flush-every 20 \
    --m3-api-base {API_BASE} --m3-cache-dir {CACHE}/zero_shot \
    --output predictions/alfa/m3_judge/zero_shot/scores.jsonl


In [ ]:
!cd {REPO} && python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores predictions/alfa/m3_judge/zero_shot/scores.jsonl \
    --score-expr "m3.p_faith * m3.p_rel" \
    --output predictions/alfa/m3_judge/zero_shot/report.json


In [ ]:
# Коммит сразу после прогона: артефакт не должен зависеть от того, доживёт ли сессия.
!cd {REPO} && git add predictions/alfa/m3_judge/zero_shot && \
    git commit -m "results(m3): судья zero_shot, локальный vLLM на A100" && \
    git log --oneline -1


## few-shot — ~20 мин

In [ ]:
# ~20 мин на 2233 кейсах
!cd {REPO} && python scripts/score.py --method m3_few_shot --variant few_shot \
    --data {DATA} --model {MODEL} --resume --flush-every 20 \
    --m3-backend openai_judge --m3-examples configs/few_shot.yaml \
    --m3-api-base {API_BASE} --m3-cache-dir {CACHE}/few_shot \
    --output predictions/alfa/m3_judge/few_shot/scores.jsonl


In [ ]:
!cd {REPO} && python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores predictions/alfa/m3_judge/few_shot/scores.jsonl \
    --score-expr "m3.p_faith * m3.p_rel" \
    --output predictions/alfa/m3_judge/few_shot/report.json


In [ ]:
# Коммит сразу после прогона: артефакт не должен зависеть от того, доживёт ли сессия.
!cd {REPO} && git add predictions/alfa/m3_judge/few_shot && \
    git commit -m "results(m3): судья few_shot, локальный vLLM на A100" && \
    git log --oneline -1


## Разделённые оси — ~20 мин

`score.py` знает только холистический промпт. Разделённые оси и self-consistency живут
в `scripts/run_m3.py` (задача C3): два независимых вызова на кейс, промпты из
`configs/prompts/*.yaml`. Он же единственный поддерживает `--concurrency`, поэтому
корпусные прогоны судьи идут здесь заметно быстрее.

In [ ]:
# ~20 мин: два вызова на кейс, но промпт relevance не получает чанков и втрое короче.
!cd {REPO} && python scripts/run_m3.py --data {DATA} \
    --output predictions/alfa/m3_judge/axes/scores.jsonl \
    --run-meta predictions/alfa/m3_judge/axes/run.yaml \
    --mode zero_shot --backend openai_judge --prompt-style axes \
    --model {MODEL} --api-base {API_BASE} \
    --cache-dir {CACHE}/axes --concurrency 16


In [ ]:
!cd {REPO} && python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores predictions/alfa/m3_judge/axes/scores.jsonl \
    --score-expr "m3.p_faith * m3.p_rel" \
    --faith-expr "m3.p_faith" --rel-expr "m3.p_rel" \
    --output predictions/alfa/m3_judge/axes/report.json


In [ ]:
!cd {REPO} && git add predictions/alfa/m3_judge/axes && \
    git commit -m "results(m3): судья с разделёнными осями, локальный vLLM" && \
    git log --oneline -1


## Self-consistency k=8 — ~50 мин

Восемь сэмплов на ось при T≈0.8. Prompt-часть переиспользуется prefix-кэшем vLLM,
дорог только выход, отсюда ×3.5 ко времени, а не ×8.

In [ ]:
# ~50 мин. Прогон резюмируется через {CACHE}/axes_sc8 — при обрыве повторить ту же команду.
!cd {REPO} && python scripts/run_m3.py --data {DATA} \
    --output predictions/alfa/m3_judge/axes_sc8/scores.jsonl \
    --run-meta predictions/alfa/m3_judge/axes_sc8/run.yaml \
    --mode zero_shot --backend openai_judge --prompt-style axes \
    --sc-n 8 --sc-temperature 0.8 \
    --model {MODEL} --api-base {API_BASE} \
    --cache-dir {CACHE}/axes_sc8 --concurrency 16


In [ ]:
!cd {REPO} && python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores predictions/alfa/m3_judge/axes_sc8/scores.jsonl \
    --score-expr "m3.p_faith * m3.p_rel" \
    --compare predictions/alfa/m3_judge/axes/scores.jsonl \
    --output predictions/alfa/m3_judge/axes_sc8/report.json


In [ ]:
!cd {REPO} && git add predictions/alfa/m3_judge/axes_sc8 && \
    git commit -m "results(m3): судья, self-consistency k=8" && \
    git log --oneline -1


## Пофрагментная верификация — ~40 мин

**Ждёт задачу D1.** `methods/m3/perchunk.py` появляется в её ветке, но точки входа в
CLI у неё в списке владения нет: `run_m3.py` принадлежит C3. Пока флага
`--prompt-style perchunk` нет, ячейка ниже упадёт с ошибкой argparse — это правильное
поведение, чинить надо CLI, а не переносить расчёт в ноутбук.

Ожидаемые ключи артефакта: `m3.max_chunk_score`, `m3.mean_chunk_score`,
`m3.chunk_disagreement` (целевая фича), `m3.n_supporting`, `m3.argmax_chunk`.
Ось только faithfulness: промпт relevance чанков не получает.

In [ ]:
# ~40 мин: 8 коротких промптов на кейс вместо одного длинного, prefix-кэш снимает общую часть.
!cd {REPO} && python scripts/run_m3.py --data {DATA} \
    --output predictions/alfa/m3_judge/perchunk/scores.jsonl \
    --run-meta predictions/alfa/m3_judge/perchunk/run.yaml \
    --mode zero_shot --backend openai_judge --prompt-style perchunk \
    --model {MODEL} --api-base {API_BASE} \
    --cache-dir {CACHE}/perchunk --concurrency 16


In [ ]:
!cd {REPO} && python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores predictions/alfa/m3_judge/perchunk/scores.jsonl \
    --score-expr "m3.max_chunk_score" \
    --output predictions/alfa/m3_judge/perchunk/report.json


In [ ]:
!cd {REPO} && git add predictions/alfa/m3_judge/perchunk && \
    git commit -m "results(m3): пофрагментная верификация" && \
    git log --oneline -1


## Выгрузка коммитов

Локальные коммиты живут на File Storage и переживают рестарт VM, но не удаление
хранилища. Пуш требует токена в переменных окружения проекта
(`GIT_ASKPASS`/credential helper) — если его нет, скачай `predictions/` через файловый
менеджер JupyterLab.

In [ ]:
!cd {REPO} && git log --oneline origin/{BRANCH}..HEAD


In [ ]:
!cd {REPO} && git push origin {BRANCH}
